In [1]:
import os
import warnings
from typing import Optional, List

# Устанавливаем прокси, если необходимо
os.environ["https_proxy"] = "http://127.0.0.1:2334"
os.environ["http_proxy"] = "http://127.0.0.1:2334"
os.environ["WANDB_PROJECT"] = "flow_llama"

import torch
import torch.nn.functional as F
from torch import nn

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    LlamaConfig,
    LlamaModel,
    LlamaPreTrainedModel,
)
from transformers.modeling_outputs import CausalLMOutputWithPast

# Подавляем некоторые предупреждения от transformers для чистоты вывода
warnings.filterwarnings("ignore", category=UserWarning)


# --- 1. Определение кастомного класса модели ---


class FlowLlamaForCausalLM(LlamaPreTrainedModel):
    """
    Кастомный класс, структурно идентичный LlamaForCausalLM, но с переопределенным
    методом forward для реализации гибридного обучения Flow Matching + Cross-Entropy.
    """

    # Копируем __init__ из LlamaForCausalLM и добавляем наши параметры
    def __init__(
        self, config: LlamaConfig, lambda_fm: float = 1.0, lambda_ce: float = 1.0
    ):
        super().__init__(config)
        self.model = LlamaModel(config)
        self.vocab_size = config.vocab_size
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

        # Сохраняем наши гиперпараметры в конфиге
        self.config.lambda_fm = lambda_fm
        self.config.lambda_ce = lambda_ce

        # Initialize weights and apply final processing
        self.post_init()

    def get_input_embeddings(self) -> nn.Embedding:
        return self.model.embed_tokens

    def set_input_embeddings(self, value: nn.Embedding):
        self.model.embed_tokens = value

    # Полностью переписанный forward без вызовов super()
    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[List[torch.FloatTensor]] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        use_cache: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
    ) -> CausalLMOutputWithPast:

        output_attentions = (
            output_attentions
            if output_attentions is not None
            else self.config.output_attentions
        )
        output_hidden_states = (
            output_hidden_states
            if output_hidden_states is not None
            else self.config.output_hidden_states
        )
        return_dict = (
            return_dict if return_dict is not None else self.config.use_return_dict
        )

        # --- Режим обучения Flow Matching (когда есть labels) ---
        if labels is not None:
            if inputs_embeds is not None:
                raise ValueError(
                    "`inputs_embeds` is not supported during training. Use `input_ids`."
                )

            embeds = self.model.embed_tokens(input_ids)
            B, S, D = embeds.shape

            # --- НАЧАЛО ИЗМЕНЕНИЙ ---

            # x0 остается прежним - это входной эмбеддинг предыдущего токена
            x0 = embeds[:, :-1, :]

            # Получаем метки для следующих токенов
            ce_labels = labels[:, 1:].clone()

            # Создаем маску для паддинга ДО того, как изменим -100
            mask = ce_labels != -100

            # Для безопасного индексирования заменяем -100 на валидный индекс (например, 0).
            # Потеря по этим позициям все равно будет занулена маской.
            safe_labels = ce_labels.clone()
            safe_labels[~mask] = 0  # Используем 0 как безопасный индекс

            # x1 - ЭТО КЛЮЧЕВОЕ ИЗМЕНЕНИЕ:
            # Берем целевые векторы из ВЫХОДНОЙ матрицы lm_head, а не из входных эмбеддингов.
            x1 = self.lm_head.weight[safe_labels]

            # --- КОНЕЦ ИЗМЕНЕНИЙ ---

            x0 = embeds[:, :-1, :]
            # x1 = embeds[:, 1:, :]
            target_velocity = x1 - x0

            t = torch.rand(B, S - 1, 1, device=embeds.device)
            xt = (1 - t) * x0 + t * x1

            flow_inputs_embeds = torch.cat([embeds[:, :1, :], xt], dim=1)

            # Прямой вызов self.model (тела трансформера)
            transformer_outputs = self.model(
                inputs_embeds=flow_inputs_embeds,
                attention_mask=attention_mask,
                position_ids=position_ids,
                past_key_values=None,  # Игнорируем кеш в режиме обучения
                use_cache=False,
                output_attentions=output_attentions,
                output_hidden_states=output_hidden_states,
                return_dict=return_dict,
            )
            hidden_states = transformer_outputs[0]

            # Вычисление нашей кастомной потери
            v_out = hidden_states[:, :-1, :]
            ce_labels = labels[:, 1:].clone()

            loss_fm = F.mse_loss(v_out, target_velocity, reduction="none").mean(dim=-1)

            logits_for_loss = self.lm_head(v_out)
            loss_ce_full = F.cross_entropy(
                logits_for_loss.reshape(-1, self.config.vocab_size),
                ce_labels.reshape(-1),
                reduction="none",
            ).reshape(B, S - 1)

            mask = ce_labels != -100
            masked_loss_fm = (loss_fm * mask).sum() / mask.sum().clamp(min=1)
            masked_loss_ce = (loss_ce_full * mask).sum() / mask.sum().clamp(min=1)
            loss = (
                self.config.lambda_fm * masked_loss_fm
                + self.config.lambda_ce * masked_loss_ce
            )

            # Для совместимости возвращаем полные логиты
            final_logits = self.lm_head(hidden_states)

        # --- Режим инференса (когда labels нет) ---
        else:
            # Прямой вызов self.model (тела трансформера)
            transformer_outputs = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                position_ids=position_ids,
                past_key_values=past_key_values,
                inputs_embeds=inputs_embeds,
                use_cache=use_cache,
                output_attentions=output_attentions,
                output_hidden_states=output_hidden_states,
                return_dict=return_dict,
            )
            hidden_states = transformer_outputs[0]
            final_logits = self.lm_head(hidden_states)
            loss = None

        if not return_dict:
            output = (final_logits,) + transformer_outputs[1:]
            return (loss,) + output if loss is not None else output

        return CausalLMOutputWithPast(
            loss=loss,
            logits=final_logits,
            past_key_values=transformer_outputs.past_key_values,
            hidden_states=transformer_outputs.hidden_states,
            attentions=transformer_outputs.attentions,
        )


# --- 2. Настройка параметров и подготовка данных ---

# Основные параметры
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
dataset_name = "wikitext"
dataset_config = "wikitext-2-raw-v1"
block_size = 128  # Длина контекста

# Загрузка токенизатора
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def tokenize_and_group(examples):
    """Функция для токенизации и группировки текста в блоки."""
    tokenized_outputs = tokenizer(examples["text"], truncation=False)

    concatenated_examples = {k: sum(v, []) for k, v in tokenized_outputs.items()}
    total_length = len(concatenated_examples[list(concatenated_examples.keys())[0]])

    total_length = (total_length // block_size) * block_size

    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result


# Загрузка и подготовка датасета
print("Loading and preparing dataset...")
raw_datasets = load_dataset(dataset_name, dataset_config)
tokenized_datasets = raw_datasets.map(
    tokenize_and_group,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    num_proc=16,
)
print(f"Dataset prepared. Train examples: {len(tokenized_datasets['train'])}")


# --- 3. Инициализация модели и настройка обучения ---

print("Initializing custom FlowLlamaForCausalLM model...")
# Загружаем конфиг от оригинальной модели
config = AutoModelForCausalLM.from_pretrained(model_name).config

# Создаем экземпляр нашего класса со случайными весами
flow_model = FlowLlamaForCausalLM(
    config=config,
    lambda_fm=1.0,  # Вес для Flow Matching loss. Требует подбора.
    lambda_ce=0.01,  # Вес для Cross-Entropy.
)

# Загружаем в нашу модель веса из предобученной TinyLlama
flow_model.load_state_dict(
    AutoModelForCausalLM.from_pretrained(model_name).state_dict()
)
print("Weights loaded from pretrained model into the custom architecture.")

# Аргументы для Trainer
training_args = TrainingArguments(
    output_dir="./flow_llama_results_v3",
    overwrite_output_dir=True,
    num_train_epochs=4,
    per_device_train_batch_size=4,  # Уменьшите, если не хватает VRAM
    gradient_accumulation_steps=1,
    save_steps=5000,
    logging_steps=50,
    learning_rate=2e-5,
    weight_decay=0.01,
    bf16=True,  # Используем смешанную точность для ускорения и экономии памяти
    # max_steps=200,  # ОГРАНИЧИВАЕМ обучение для быстрого теста! Уберите для полноценного обучения.
)

# Data Collator для создания батчей с паддингом
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Инициализация Trainer
trainer = Trainer(
    model=flow_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    data_collator=data_collator,
)

# Запуск обучения
print("\n--- Starting Training ---")
trainer.train()
print("Training finished.")


# --- 4. Демонстрация генерации ---

Loading and preparing dataset...
Dataset prepared. Train examples: 22328
Initializing custom FlowLlamaForCausalLM model...
Weights loaded from pretrained model into the custom architecture.
is_torch_bf16_gpu_available True


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.



--- Starting Training ---


wandb: Currently logged in as: dimweb to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
50,0.539600
100,0.343300
150,0.340500
200,0.338600
250,0.336200
300,0.334700
350,0.332600
400,0.330500
450,0.328300
500,0.327400


Training finished.


In [5]:
# --- 4. Демонстрация генерации (ИСПРАВЛЕННАЯ ВЕРСИЯ) ---
@torch.no_grad()
def generate_token_by_token_flow(
    model: FlowLlamaForCausalLM,
    tokenizer: AutoTokenizer,
    prompt: str,
    max_new_tokens: int = 20,
    num_flow_steps: int = 10,
) -> str:
    """Генерирует текст с помощью итеративного flow-процесса для каждого токена."""
    model.eval()
    device = model.device

    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    # Получаем доступ к слоям эмбеддингов и головы
    input_embed_layer = model.get_input_embeddings()
    output_head_layer = model.lm_head

    context_embeds = input_embed_layer(input_ids)

    dt = 1.0 / num_flow_steps

    for _ in range(max_new_tokens):
        x0 = context_embeds[:, -1:, :]

        current_x = x0
        for i in range(num_flow_steps):
            t_step = i * dt
            current_input_embeds = torch.cat([context_embeds, current_x], dim=1)

            # Вызываем .model, чтобы получить hidden_states
            outputs = model.model(inputs_embeds=current_input_embeds)
            hidden_states = outputs[0]
            v_out = hidden_states[:, -1:, :]

            # Шаг по ОДУ (метод Эйлера)
            current_x = current_x + v_out * dt

        # x1_pred - результат нашего flow-процесса.
        # Этот вектор находится в пространстве ВЫХОДНЫХ эмбеддингов.
        x1_pred = current_x

        # --- КЛЮЧЕВОЕ ИСПРАВЛЕНИЕ ---
        # Сравниваем x1_pred с правильной матрицей - lm_head.weight
        all_output_embeds = output_head_layer.weight
        logits = torch.matmul(x1_pred.squeeze(1), all_output_embeds.t())

        # Находим ID токена с максимальным логитом (максимальным сходством)
        next_token_id = torch.argmax(logits, dim=-1).unsqueeze(0)

        # Добавляем ID нового токена к последовательности
        input_ids = torch.cat([input_ids, next_token_id], dim=-1)

        # Для следующего шага нам нужен ВХОДНОЙ эмбеддинг нового токена
        next_token_input_embed = input_embed_layer(next_token_id)
        context_embeds = torch.cat([context_embeds, next_token_input_embed], dim=1)

        if next_token_id.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


print("\n--- Starting Generation Demo ---")
# prompt = "The capital of France is"
# prompt = "The capital of Russia is"
# prompt = "The capital of Russia is"
# prompt = "The game began development in 2010"
# prompt = "For several years the arsenal"
# prompt = "The item was intended simply as a piece"
prompt = "The item was intended simply as a piece"
generated_text = generate_token_by_token_flow(
    model=flow_model, tokenizer=tokenizer, prompt=prompt, num_flow_steps=50
)
print(f"Prompt: {prompt}")
print(f"Generated: {generated_text}")


--- Starting Generation Demo ---
Prompt: The item was intended simply as a piece
Generated: The item was intended simply as a piece the the the the the the the the " the " the " the " the " the " the


In [27]:
raw_datasets["train"][65]

{'text': ' The item was intended simply as a piece of news , but telegraph lines quickly spread the news throughout the state , fueling procession sentiment . The rumor was interpreted by some Arkansans as a call from the governor to assemble to help expel the federal troops from the arsenal . By February 5 , six militia units , consisting of 1 @,@ 000 men , with a guarantee that the numbers could be increased to 5 @,@ 000 if the situations deemed it necessary , had assembled in Little Rock . Governor Rector vehemently denied ordering the troops to assemble or giving any order at all in connection with the troops . Faced with the fact that the military had assembled believing they were following his orders and the consensus of the citizens of Little Rock against any armed conflict between the civilian army and federal troops , Governor Rector was forced to take control of the situation . On February 6 , he sent a formal demand for surrender of the arsenal to Captain Totten , \n'}

In [ ]:
# весьма вероятно это не работает как надо потому что у нас эмбединги должны быть зафиксированы. мы же vae не обучаем когда тренируем диффузионную модель.